In [1]:
import openfhe_numpy as onp
from openfhe import *
import numpy as np

In [2]:
def decrypt_and_print(entity, keys):
    decrypted = entity.decrypt(keys.secretKey, unpack_type="original")
    print(decrypted)

In [3]:
mult_depth = 10

params = CCParamsCKKSRNS()
params.SetMultiplicativeDepth(mult_depth)
params.SetScalingModSize(59)
params.SetFirstModSize(60)
params.SetScalingTechnique(FIXEDAUTO)
params.SetKeySwitchTechnique(HYBRID)
params.SetSecretKeyDist(UNIFORM_TERNARY)

cc = GenCryptoContext(params)
cc.Enable(PKESchemeFeature.PKE)
cc.Enable(PKESchemeFeature.LEVELEDSHE)
cc.Enable(PKESchemeFeature.ADVANCEDSHE)

keys = cc.KeyGen()
cc.EvalMultKeyGen(keys.secretKey)
cc.EvalSumKeyGen(keys.secretKey)

batch_size = cc.GetRingDimension() // 2
print("\n****** CRYPTO PARAMETERS ******")
print(f"Total Slots: {batch_size}")
print("*******************************")


****** CRYPTO PARAMETERS ******
Total Slots: 32768
*******************************


In [4]:
print("\nInput")
print("\nMatrix:\n", matrix, matrix.shape)
print("\nVector:\n", vector, vector.shape)

print("\n" + "*" * 60)
print(f"* Homomorphic Matrix Vector Product")
print("*" * 60)

expected = matrix @ vector
print(f"\nExpected:\n{expected}")


Input


NameError: name 'matrix' is not defined

In [22]:
ctm_m_rm = onp.array(
    cc=cc,
    data=matrix,
    batch_size=batch_size,
    order=onp.ROW_MAJOR,
    mode="tile",
    fhe_type="C",
    public_key=keys.publicKey,
)
ctm_m_rm.extra["colkey"] = onp.sum_col_keys(keys.secretKey)
ctv_v_cm = onp.array(
    cc=cc,
    data=vector,
    batch_size=batch_size,
    order=onp.COL_MAJOR,
    mode="tile",
    fhe_type="C",
    public_key=keys.publicKey,
)

In [368]:
def slice_vector(enc_vec, start, end, keys):
    """
    Slice an encrypted vector while keeping it encrypted using matrix multiplication.
    
    Treat the vector of length n as a matrix of shape (1, n).
    Create a selection mask in numpy which is 0 everywhere except for the range [start, end)
    Matrix multiply the selection matrix with the original vector to get the sliced result. 
    
    Parameters
    ----------
    enc_vec :  CTArray
        The encrypted vector to slice (in ROW_MAJOR format)
    start : int
        Start index (inclusive)
    end : int
        End index (exclusive)
    keys:
        The keys generated by the cryptocontext (cc.KeyGen())
    
    Returns
    -------
    CTArray
        The sliced encrypted vector containing elements from index start to end-1
    """
    slice_length = end - start
    length = enc_vec.original_shape[0]

    selection_matrix = np.zeros((slice_length, length))
    for i in range(slice_length):
        selection_matrix[i, start + i] = 1.0

    enc_selection = onp.array(
        cc=enc_vec.data.GetCryptoContext(),
        data=selection_matrix,
        batch_size=enc_vec.batch_size,
        order=onp.ROW_MAJOR,
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey,
    )
    enc_selection.extra["colkey"] = onp.sum_col_keys(keys.secretKey) # maybe this can be moved outside so only public key would be needed in this function

    slice_result = enc_selection @ enc_vec

    return slice_result

In [78]:
ohe_matrix = np.array([[0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1]], dtype=np.float32)
ohe_matrix

array([[0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1.]], dtype=float32)

In [45]:
start1, end1 = 0, 8
start2, end2 = 8, 12

In [46]:
cols1 = ohe_matrix[:, start1:end1]
cols2 = ohe_matrix[:, start2:end2]

In [47]:
cols1, cols2

(array([[0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0.]], dtype=float32),
 array([[0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 1., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]], dtype=float32))

In [49]:
cols1.shape, cols2.shape

((5, 8), (5, 4))

In [54]:
print((cols1.T @ cols2).flatten())

[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 1. 0. 0. 0. 0. 0.]


In [66]:
def select(matrix, start, end):
    n_cols = matrix.shape[-1]
    n_selected = end - start
    
    # Create selection matrix: (n_cols, n_selected)
    # Each column j has a 1 at position (start + j)
    selection_matrix = np.zeros((n_cols, n_selected))
    for i in range(n_selected):
        selection_matrix[start + i, i] = 1.0
    
    # Matrix multiplication: (rows, n_cols) @ (n_cols, n_selected) = (rows, n_selected)
    return matrix @ selection_matrix

In [67]:
select(ohe_matrix, 0, 8)

array([[0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0.]])

In [79]:
enc_matrix = onp.array(
        cc=cc,
        data=ohe_matrix,
        batch_size=batch_size,
        order=onp.COL_MAJOR,
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey,
    )

In [91]:
def select_columns(enc_matrix, start, end, keys):
    """
    Select columns from an encrypted matrix while keeping it encrypted using matrix multiplication.
    
    Parameters
    ----------
    enc_matrix : CTArray
        The encrypted matrix to select columns from (in ROW_MAJOR format)
        Shape: (rows, n_cols)
    start : int
        Start column index (inclusive)
    end : int
        End column index (exclusive)
    keys : KeyPair
        The keys generated by the cryptocontext (cc.KeyGen())
    
    Returns
    -------
    CTArray
        The matrix with selected columns, shape (rows, end-start)
    """
    n_cols = enc_matrix.original_shape[-1]
    n_selected = end - start
    
    # Create selection matrix: (n_cols, n_selected)
    # Each column j has a 1 at position (start + j)
    selection_matrix = np.zeros((n_cols, n_selected))
    for j in range(n_selected):
        selection_matrix[start + j, j] = 1.0
    
    # Encrypt selection matrix with COMPLEMENTARY encoding
    # Matrix is ROW_MAJOR, so if enc_matrix is ROW_MAJOR, use COL_MAJOR here
    enc_selection = onp.array(
        cc=enc_matrix.data.GetCryptoContext(),
        data=selection_matrix,
        batch_size=enc_matrix.batch_size,
        order=onp.COL_MAJOR,  # Complementary to enc_matrix's ROW_MAJOR
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey,
    )
    
    # Attach column sum key (needed for matrix-matrix multiplication)
    enc_selection.extra["colkey"] = onp.sum_col_keys(keys.secretKey)
    
    # Matrix multiplication: (rows, n_cols) @ (n_cols, n_selected) = (rows, n_selected)
    result = enc_matrix @ enc_selection
    decrypt_and_print(result, keys)
    return result


## Matrix multiplication

In [4]:
mat = np.arange(1, 16).reshape(3, 5)
mat

array([[ 1,  2,  3,  4,  5],
       [ 6,  7,  8,  9, 10],
       [11, 12, 13, 14, 15]])

In [5]:
start1, end1 = 0, 2
start2, end2 = 2, 5

In [6]:
mat[:, start1:end1]

array([[ 1,  2],
       [ 6,  7],
       [11, 12]])

In [7]:
mat[:, start2:end2]

array([[ 3,  4,  5],
       [ 8,  9, 10],
       [13, 14, 15]])

In [8]:
i = 0

In [9]:
def select_column(mat, i):
    """
    This only uses Matrix vector multiplications
    """
    selection_vector = np.zeros(mat.shape[1])
    selection_vector[i] = 1.0
    # encrypt selection_vector
    return mat @ selection_vector

In [10]:
dot_prod = []

for i in range(start1, end1):
    this_row = []
    for j in range(start2, end2):
        col1 = select_column(mat, i)
        col2 = select_column(mat, j)
        this_row.append(np.dot(col1, col2))
    dot_prod.append(this_row)

dot_prod = np.array(dot_prod)

In [11]:
dot_prod

array([[194., 212., 230.],
       [218., 239., 260.]])

In [12]:
mat[:, start1:end1].T @ mat[:, start2:end2]

array([[194, 212, 230],
       [218, 239, 260]])

In [13]:
cls = {}
cls['age1'] = (start1, end1)
cls['bmi1'] = (start2, end2)
cls

{'age1': (0, 2), 'bmi1': (2, 5)}

In [14]:
attr1, attr2 = 'age1', 'bmi1'    

In [171]:
def compute_twoway(mat, attr1, attr2):
    """
    Need select_column
    """
    start1, end1 = cls[attr1]
    start2, end2 = cls[attr2]

    dot_prod = []
    
    for i in range(start1, end1):
        this_row = []
        for j in range(start2, end2):
            col1 = select_column(mat, i)
            col2 = select_column(mat, j)
            this_row.append(np.dot(col1, col2))
        dot_prod.append(this_row)

    dot_prod = np.array(dot_prod)

    return dot_prod.flatten()

In [172]:
compute_twoway(mat, attr1, attr2)

array([194., 212., 230., 218., 239., 260.])

## Now lets see what the other slo

In [140]:
mat_enc = onp.array(
        cc=cc,
        data=mat,
        batch_size=batch_size,
        order=onp.ROW_MAJOR,  # Complementary to enc_matrix's ROW_MAJOR
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey,
    )
mat_enc.extra["colkey"] = onp.sum_col_keys(keys.secretKey) # this works

In [187]:
def select_column_he(mat_enc, column):
    selection_vector = np.zeros(mat_enc.original_shape[1])
    selection_vector[column] = 1.0
    
    # encrypt selection_vector
    selection_vector_enc = onp.array(
            cc=cc,
            data=selection_vector,
            batch_size=mat_enc.batch_size,
            order=onp.COL_MAJOR,
            mode="tile",
            fhe_type="C",
            public_key=keys.publicKey,
        )
    selected_column_enc =  mat_enc @ selection_vector_enc

    return selected_column_enc

In [179]:
column = 2
selected_column_enc = select_column_he(mat_enc, column)

In [180]:
selected_column_enc.decrypt(keys.secretKey, unpack_type="original"), mat[:, column]

(array([ 3.,  8., 13.]), array([ 3,  8, 13]))

In [181]:
col1_enc = select_column_he(mat_enc, column=1)
col2_enc = select_column_he(mat_enc, column=2)

In [182]:
pdt_enc = col1_enc @ col2_enc

In [240]:
col1_enc.decrypt(keys.secretKey, unpack_type="original")

array([ 2.,  7., 12.])

In [241]:
col2_enc.decrypt(keys.secretKey, unpack_type="original")

array([ 3.,  8., 13.])

In [246]:
pdt_enc = (col1_enc @ col2_enc) * (1 / (batch_size // 4))
# pdt_enc = onp.sum(col1_enc * col2_enc)

In [247]:
pdt_enc.decrypt(keys.secretKey, unpack_type="original")

217.9999999861199

In [248]:
def dot_product(vec1, vec2):
    dot_product_enc = (vec1 @ vec2) * (1 / (vec1.batch_size // 4))
    # print(f"Multiplying {vec1.decrypt(keys.secretKey, unpack_type="original")} and {vec2.decrypt(keys.secretKey, unpack_type="original")}")
    return dot_product_enc

In [313]:
from openfhe_numpy.tensor.ctarray import CTArray
from openfhe_numpy import ArrayEncodingType

def transform_marginal(marginal_enc):
    size = len(marginal_enc)
    sz = 1
    while sz < size:
        sz *= 2
    new_marginal = []
    for i in range(len(marginal_enc)):
        new_marginal.append(CTArray(
            marginal_enc[i].data, marginal_enc[0].shape, (sz, 1), selection_vector_enc.shape, ArrayEncodingType.ROW_MAJOR
        ))

    return new_marginal

In [349]:
def compute_twoway_he(enc_mat, attr1, attr2):
    """
    Need select_column
    """
    start1, end1 = cls[attr1]
    start2, end2 = cls[attr2]

    dot_prod = []
    
    for i in range(start1, end1):
        col1 = select_column_he(enc_mat, i)
        for j in range(start2, end2):
            col2 = select_column_he(enc_mat, j)
            dot_prod.append(dot_product(col1, col2))

    marginal = np.zeros(len(dot_prod))
    marginal_enc_cum = onp.array(
            cc=cc,
            data=marginal,
            batch_size=enc_mat.batch_size,
            order=onp.ROW_MAJOR,
            mode="tile",
            fhe_type="C",
            public_key=keys.publicKey,
        )

    marginal_transformed_enc = transform_marginal(dot_prod)

    for i in range(len(marginal_transformed_enc)):
        selection_vector = np.zeros(len(marginal_transformed_enc))
        selection_vector[i] = 1.0
        # encrypt it
        selection_vector_enc = onp.array(
            cc=cc,
            data=selection_vector,
            # data = scalar,
            batch_size=mat_enc.batch_size,
            order=onp.ROW_MAJOR,
            mode="tile",
            fhe_type="C",
            public_key=keys.publicKey,
        )
        
        marginal_enc_cum += selection_vector_enc * marginal_transformed_enc[i]

    return marginal_enc_cum

In [350]:
marginal_enc = compute_twoway_he(mat_enc, attr1, attr2)

In [351]:
marginal_enc.decrypt(keys.secretKey, unpack_type="original")

array([193.99999999, 211.99999999, 229.99999999, 217.99999999,
       238.99999999, 259.99999999])

In [352]:
compute_twoway(mat, attr1, attr2)

array([194., 212., 230., 218., 239., 260.])

In [365]:
mat_enc.extra["rowkey"] = onp.sum_row_keys(keys.secretKey, mat_enc.ncols, mat_enc.batch_size)
onp.sum(mat_enc, axis=0).decrypt(keys.secretKey, unpack_type="original") * (1/1024)

array([18., 21., 24., 27., 30.])

In [360]:
np.sum(mat, axis=0)

array([18, 21, 24, 27, 30])

In [357]:
mat[:, 0].sum(), mat[:, 1].sum(), mat[:, 2].sum()

(np.int64(18), np.int64(21), np.int64(24))

In [366]:
def compute_oneway_he(enc_matrix, attr):
    start, end = cls[attr]

    enc_matrix.extra["rowkey"] = onp.sum_row_keys(keys.secretKey, enc_matrix.ncols, enc_matrix.batch_size)
    oneway = onp.sum(mat_enc, axis=0) * (1/1024)

    return slice_vector(oneway, start, end, keys)

In [369]:
compute_oneway_he(mat_enc, 'age1').decrypt(keys.secretKey, unpack_type="original")

array([18., 21.])